# Customer Churn: Logistic Regression & LightGBM

EDA, preprocessing, baseline comparison, hyperparameter tuning, threshold optimization,
early stopping and SHAP. See `README.md` for setup and method limitations.

Place your dataset at `data/Churn_Modelling.csv`, then run all cells in order.
Outputs are intentionally cleared for Git. Generated files go to `artifacts/`.

In [ ]:
# 1. Import libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from pathlib import Path
from IPython.display import display
import joblib

# Configuration (paths are relative to this notebook's directory)
DATA_PATH = Path("data/Churn_Modelling.csv")
ARTIFACT_DIR = Path("artifacts")
N_ITER = 40
CV = 5
RUN_SHAP = True
MODEL_JOBS = -1
SEARCH_JOBS = 1

In [ ]:
# 2. Load dataset
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}. "
        "Place Churn_Modelling.csv in data/ or update DATA_PATH. See data/README.md."
    )
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# 3. Check dataset shape and columns

print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# 4. Check data information
df.info()

In [ ]:
# 5. Check missing values
df.isnull().sum()

In [ ]:
# 6. Check duplicated rows and target distribution
print("Number of duplicated rows:")
print(df.duplicated().sum())

print("\nTarget counts:")
print(df["Exited"].value_counts())

print("\nTarget percentages:")
print(df["Exited"].value_counts(normalize=True) * 100)

In [ ]:
# 7. Basic statistical summary

print("Numerical summary:")
display(df.describe())
print("\nCategorical summary:")
display(df.describe(include="object"))

In [ ]:
# 8. Visualize target distribution

plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="Exited"
)
plt.title("Target Distribution: Customer Churn")
plt.xlabel("Exited")
plt.ylabel("Count")
plt.show()

# EDA

In [ ]:
# 9. Analyze categorical features by churn

categorical_cols = ["Geography", "Gender"]

for col in categorical_cols:
    print(f"\nChurn summary by {col}:")

    churn_summary = df.groupby(col)["Exited"].agg(
        total_customers="count",
        churn_customers="sum",
        churn_rate="mean"
    )

    churn_summary["churn_rate"] = churn_summary["churn_rate"] * 100
    churn_summary = churn_summary.sort_values(by="churn_rate", ascending=False)

    display(churn_summary)

    plt.figure(figsize=(6, 4))

    sns.countplot(
        data=df,
        x=col,
        hue="Exited"
    )

    plt.title(f"Churn Distribution by {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.legend(title="Exited", labels=["Not Churn", "Churn"])
    plt.show()

In [ ]:
# 10. Analyze numerical features by churn

numerical_cols = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary"
]

for col in numerical_cols:
    print(f"\nSummary statistics for {col} by churn:")

    summary = df.groupby("Exited")[col].agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )

    display(summary)

    plt.figure(figsize=(7, 4))

    sns.histplot(
        data=df,
        x=col,
        hue="Exited",
        kde=True,
        bins=30
    )

    plt.title(f"Distribution of {col} by Churn")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
# 11. EDA: Analyze binary features by churn

binary_cols = [
    "HasCrCard",
    "IsActiveMember"
]

for col in binary_cols:
    print(f"\nChurn summary by {col}:")

    summary = df.groupby(col)["Exited"].agg(
        total_customers="count",
        churn_customers="sum",
        churn_rate="mean"
    )

    summary["churn_rate"] = summary["churn_rate"] * 100
    summary = summary.sort_values(by="churn_rate", ascending=False)

    display(summary)

    plt.figure(figsize=(6, 4))

    sns.countplot(
        data=df,
        x=col,
        hue="Exited"
    )

    plt.title(f"Churn Distribution by {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.legend(title="Exited", labels=["Not Churn", "Churn"])
    plt.show()

In [ ]:
# 12. EDA: Correlation heatmap

# Select numerical columns for correlation analysis
correlation_cols = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

# Calculate correlation matrix
corr_matrix = df[correlation_cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

# Data Preprocessing

In [ ]:
# 13. Drop unnecessary columns

columns_to_drop = [
    "RowNumber",
    "CustomerId",
    "Surname"
]

df_model = df.drop(columns=columns_to_drop)

df_model.head()

In [ ]:
# 14. Define features and target

X = df_model.drop(columns=["Exited"])
y = df_model["Exited"]

print("Feature matrix shape:")
print(X.shape)
print("\nTarget vector shape:")
print(y.shape)
print("\nFeatures:")
print(X.columns.tolist())
print("\nTarget:")
print(y.name)

In [ ]:
# 15. Split data into training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
# 16. Identify categorical and numerical features

categorical_features = [
    "Geography",
    "Gender"
]

numerical_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

**Logistic Regression**

In [ ]:
# 17. Create preprocessing pipeline for Logistic Regression

# Create OneHotEncoder
# handle_unknown="ignore" prevents errors if test data contains unseen categories
encoder = OneHotEncoder(handle_unknown="ignore")

# Create preprocessing pipeline for Logistic Regression
logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", encoder, categorical_features)
    ]
)

In [ ]:
# 18. Train Logistic Regression baseline model
# Create Logistic Regression pipeline
logistic_model = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

# Train the model using training data
logistic_model.fit(X_train, y_train)

In [ ]:
# 19. Evaluate Logistic Regression baseline model

# Predict class labels on the test set
y_pred_logistic = logistic_model.predict(X_test)

# Predict churn probabilities on the test set
y_proba_logistic = logistic_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic)
recall_logistic = recall_score(y_test, y_pred_logistic)
f1_logistic = f1_score(y_test, y_pred_logistic)
roc_auc_logistic = roc_auc_score(y_test, y_proba_logistic)

print("Logistic Regression Baseline Results")
print("=" * 45)
print(f"Accuracy:  {accuracy_logistic:.4f}")
print(f"Precision: {precision_logistic:.4f}")
print(f"Recall:    {recall_logistic:.4f}")
print(f"F1-score:  {f1_logistic:.4f}")
print(f"ROC-AUC:   {roc_auc_logistic:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

# Confusion matrix
cm_logistic = confusion_matrix(y_test, y_pred_logistic)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_logistic,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

**Light GBM**

In [ ]:
# 20. Create preprocessing pipeline for LightGBM
# Create OneHotEncoder for categorical variables
lgbm_encoder = OneHotEncoder(handle_unknown="ignore")

# Create preprocessing pipeline for LightGBM
lgbm_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", lgbm_encoder, categorical_features)
    ]
)

In [ ]:
# 21. Train LightGBM model

from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

lgbm_model = Pipeline(
    steps=[
        ("preprocessor", lgbm_preprocessor),
        ("model", LGBMClassifier(
            objective="binary",
            class_weight="balanced",
            random_state=42,
            n_jobs=MODEL_JOBS,
            verbosity=-1
        ))
    ]
)

lgbm_model.fit(X_train, y_train)

In [ ]:
# 22. Evaluate LightGBM model

# Predict class labels on the test set
y_pred_lgbm = lgbm_model.predict(X_test)

# Predict churn probabilities on the test set
y_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
precision_lgbm = precision_score(y_test, y_pred_lgbm)
recall_lgbm = recall_score(y_test, y_pred_lgbm)
f1_lgbm = f1_score(y_test, y_pred_lgbm)
roc_auc_lgbm = roc_auc_score(y_test, y_proba_lgbm)

print("LightGBM Model Results")
print("=" * 45)
print(f"Accuracy:  {accuracy_lgbm:.4f}")
print(f"Precision: {precision_lgbm:.4f}")
print(f"Recall:    {recall_lgbm:.4f}")
print(f"F1-score:  {f1_lgbm:.4f}")
print(f"ROC-AUC:   {roc_auc_lgbm:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm))

# Confusion matrix
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - LightGBM")
plt.show()

In [ ]:
# 23. Compare Logistic Regression and LightGBM results

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LightGBM"
    ],
    "Accuracy": [
        accuracy_logistic,
        accuracy_lgbm
    ],
    "Precision": [
        precision_logistic,
        precision_lgbm
    ],
    "Recall": [
        recall_logistic,
        recall_lgbm
    ],
    "F1-score": [
        f1_logistic,
        f1_lgbm
    ],
    "ROC-AUC": [
        roc_auc_logistic,
        roc_auc_lgbm
    ]
})

model_comparison

# Hyperparameter tuning

In [ ]:
# 24. Tune LightGBM hyperparameters using RandomizedSearchCV

from sklearn.model_selection import RandomizedSearchCV

# Create a new LightGBM pipeline for tuning
lgbm_tuning_model = Pipeline(
    steps=[
        ("preprocessor", lgbm_preprocessor),
        ("model", LGBMClassifier(
            objective="binary",
            class_weight="balanced",
            random_state=42,
            n_jobs=MODEL_JOBS,
            verbosity=-1
        ))
    ]
)

# Define hyperparameter search space
param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "model__num_leaves": [15, 31, 45, 63],
    "model__max_depth": [3, 5, 7, -1],
    "model__min_child_samples": [10, 20, 30, 50, 70],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.01, 0.1, 0.5, 1],
    "model__reg_lambda": [0, 0.01, 0.1, 0.5, 1]
}

# Set up RandomizedSearchCV
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm_tuning_model,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="roc_auc",
    cv=CV,
    random_state=42,
    n_jobs=SEARCH_JOBS,
    verbose=1
)

# Train and tune only on the training set
random_search_lgbm.fit(X_train, y_train)

print("Best ROC-AUC score from cross-validation:")
print(random_search_lgbm.best_score_)

print("\nBest hyperparameters:")
print(random_search_lgbm.best_params_)

In [ ]:
# 25. Evaluate tuned LightGBM model

# Get the best LightGBM pipeline from RandomizedSearchCV
best_lgbm_model = random_search_lgbm.best_estimator_

# Predict class labels on the test set
y_pred_lgbm_tuned = best_lgbm_model.predict(X_test)

# Predict churn probabilities on the test set
y_proba_lgbm_tuned = best_lgbm_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy_lgbm_tuned = accuracy_score(y_test, y_pred_lgbm_tuned)
precision_lgbm_tuned = precision_score(y_test, y_pred_lgbm_tuned)
recall_lgbm_tuned = recall_score(y_test, y_pred_lgbm_tuned)
f1_lgbm_tuned = f1_score(y_test, y_pred_lgbm_tuned)
roc_auc_lgbm_tuned = roc_auc_score(y_test, y_proba_lgbm_tuned)

print("Tuned LightGBM Model Results")
print("=" * 45)
print(f"Accuracy:  {accuracy_lgbm_tuned:.4f}")
print(f"Precision: {precision_lgbm_tuned:.4f}")
print(f"Recall:    {recall_lgbm_tuned:.4f}")
print(f"F1-score:  {f1_lgbm_tuned:.4f}")
print(f"ROC-AUC:   {roc_auc_lgbm_tuned:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm_tuned))

# Confusion matrix
cm_lgbm_tuned = confusion_matrix(y_test, y_pred_lgbm_tuned)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm_tuned,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Tuned LightGBM")
plt.show()

In [ ]:
# 26. Compare all model results

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LightGBM Default",
        "LightGBM Tuned"
    ],
    "Accuracy": [
        accuracy_logistic,
        accuracy_lgbm,
        accuracy_lgbm_tuned
    ],
    "Precision": [
        precision_logistic,
        precision_lgbm,
        precision_lgbm_tuned
    ],
    "Recall": [
        recall_logistic,
        recall_lgbm,
        recall_lgbm_tuned
    ],
    "F1-score": [
        f1_logistic,
        f1_lgbm,
        f1_lgbm_tuned
    ],
    "ROC-AUC": [
        roc_auc_logistic,
        roc_auc_lgbm,
        roc_auc_lgbm_tuned
    ]
})

model_comparison

In [ ]:
# 27. Find the best threshold for Tuned LightGBM using a validation set

from sklearn.base import clone

# Split the original training set into a smaller training set and a validation set
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# Clone the tuned LightGBM model so we can retrain it only on X_train_inner
lgbm_threshold_model = clone(best_lgbm_model)

# Train the cloned tuned model on the inner training set
lgbm_threshold_model.fit(X_train_inner, y_train_inner)

# Predict churn probabilities on the validation set
y_val_proba = lgbm_threshold_model.predict_proba(X_val)[:, 1]

# Try different threshold values
thresholds = np.arange(0.10, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

# Sort thresholds by F1-score
threshold_results_df = threshold_results_df.sort_values(
    by="F1-score",
    ascending=False
)

threshold_results_df.head(10)

In [ ]:
# 28. Evaluate Tuned LightGBM using the selected threshold

# Select the best threshold based on validation F1-score
best_threshold = threshold_results_df.iloc[0]["Threshold"]

print("Selected threshold:")
print(best_threshold)

# Predict churn probabilities on the test set using the tuned LightGBM model
y_proba_lgbm_tuned_threshold = best_lgbm_model.predict_proba(X_test)[:, 1]

# Convert probabilities into class labels using the selected threshold
y_pred_lgbm_tuned_threshold = (y_proba_lgbm_tuned_threshold >= best_threshold).astype(int)

# Calculate evaluation metrics
accuracy_lgbm_tuned_threshold = accuracy_score(y_test, y_pred_lgbm_tuned_threshold)
precision_lgbm_tuned_threshold = precision_score(y_test, y_pred_lgbm_tuned_threshold)
recall_lgbm_tuned_threshold = recall_score(y_test, y_pred_lgbm_tuned_threshold)
f1_lgbm_tuned_threshold = f1_score(y_test, y_pred_lgbm_tuned_threshold)
roc_auc_lgbm_tuned_threshold = roc_auc_score(y_test, y_proba_lgbm_tuned_threshold)

print("Tuned LightGBM with Optimized Threshold Results")
print("=" * 55)
print(f"Threshold:  {best_threshold:.2f}")
print(f"Accuracy:   {accuracy_lgbm_tuned_threshold:.4f}")
print(f"Precision:  {precision_lgbm_tuned_threshold:.4f}")
print(f"Recall:     {recall_lgbm_tuned_threshold:.4f}")
print(f"F1-score:   {f1_lgbm_tuned_threshold:.4f}")
print(f"ROC-AUC:    {roc_auc_lgbm_tuned_threshold:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm_tuned_threshold))

# Confusion matrix
cm_lgbm_tuned_threshold = confusion_matrix(y_test, y_pred_lgbm_tuned_threshold)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm_tuned_threshold,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Tuned LightGBM with Optimized Threshold")
plt.show()

In [ ]:
# 29. Final comparison of all models

final_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LightGBM Default",
        "LightGBM Tuned",
        f"LightGBM Tuned + Threshold {best_threshold:.2f}"
    ],
    "Accuracy": [
        accuracy_logistic,
        accuracy_lgbm,
        accuracy_lgbm_tuned,
        accuracy_lgbm_tuned_threshold
    ],
    "Precision": [
        precision_logistic,
        precision_lgbm,
        precision_lgbm_tuned,
        precision_lgbm_tuned_threshold
    ],
    "Recall": [
        recall_logistic,
        recall_lgbm,
        recall_lgbm_tuned,
        recall_lgbm_tuned_threshold
    ],
    "F1-score": [
        f1_logistic,
        f1_lgbm,
        f1_lgbm_tuned,
        f1_lgbm_tuned_threshold
    ],
    "ROC-AUC": [
        roc_auc_logistic,
        roc_auc_lgbm,
        roc_auc_lgbm_tuned,
        roc_auc_lgbm_tuned_threshold
    ]
})

final_model_comparison

In [ ]:
# 30. Extract feature names after preprocessing

# Get the fitted preprocessor from the final tuned LightGBM model
fitted_preprocessor = best_lgbm_model.named_steps["preprocessor"]

# Get numerical feature names
num_feature_names = numerical_features

# Get categorical feature names after One-Hot Encoding
cat_feature_names = fitted_preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)

# Combine numerical and categorical feature names
feature_names = list(num_feature_names) + list(cat_feature_names)

feature_names

In [ ]:
# 31. Get LightGBM feature importance

# Get the trained LightGBM model from the final pipeline
trained_lgbm_model = best_lgbm_model.named_steps["model"]

# Create a DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": trained_lgbm_model.feature_importances_
})

# Sort features by importance
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Show top 15 most important features
feature_importance_df.head(12)

In [ ]:
# 32. Visualize LightGBM feature importance

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance_df,
    x="Importance",
    y="Feature"
)

plt.title("LightGBM Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
# Optional: Explain feature impact using SHAP
if RUN_SHAP:
    import shap

    X_test_transformed = fitted_preprocessor.transform(X_test)
    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()
    X_test_transformed_df = pd.DataFrame(
        X_test_transformed, columns=feature_names
    )
    explainer = shap.TreeExplainer(trained_lgbm_model)
    shap_values = explainer.shap_values(X_test_transformed_df)
    # Older SHAP versions may return one array per class.
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    elif np.ndim(shap_values) == 3:
        shap_values = shap_values[:, :, 1]
    shap.summary_plot(shap_values, X_test_transformed_df)
else:
    print("SHAP skipped (RUN_SHAP=False).")

# Fine-tuning with early stopping

In [ ]:
# 35. Fine-tune LightGBM with early stopping

from sklearn.base import clone
from lightgbm import early_stopping, log_evaluation

# Split the training set into inner training and validation sets
X_train_ft, X_val_ft, y_train_ft, y_val_ft = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# Clone the LightGBM preprocessor
preprocessor_ft = clone(lgbm_preprocessor)

# Fit the preprocessor only on the inner training set
X_train_ft_transformed = preprocessor_ft.fit_transform(X_train_ft)

# Transform the validation set using the fitted preprocessor
X_val_ft_transformed = preprocessor_ft.transform(X_val_ft)

# Extract the best parameters from previous RandomizedSearchCV
best_params_clean = {
    key.replace("model__", ""): value
    for key, value in random_search_lgbm.best_params_.items()
}

# Use a high number of trees and let early stopping decide the best number
best_params_clean["n_estimators"] = 2000

# Create the fine-tuned LightGBM model
lgbm_finetuned_model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=42,
    n_jobs=MODEL_JOBS,
    verbosity=-1,
    **best_params_clean
)

# Train with early stopping
lgbm_finetuned_model.fit(
    X_train_ft_transformed,
    y_train_ft,
    eval_set=[(X_val_ft_transformed, y_val_ft)],
    eval_metric="auc",
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

print("Best iteration found by early stopping:")
print(lgbm_finetuned_model.best_iteration_)

In [ ]:
# 36. Retrain fine-tuned LightGBM on the full training set

# Get the best number of trees found by early stopping
best_n_estimators = lgbm_finetuned_model.best_iteration_

print("Best number of estimators from early stopping:")
print(best_n_estimators)

# Create a new preprocessor for the final fine-tuned model
final_preprocessor_ft = clone(lgbm_preprocessor)

# Fit the preprocessor on the full training set
X_train_transformed_ft = final_preprocessor_ft.fit_transform(X_train)

# Transform the test set using the fitted preprocessor
X_test_transformed_ft = final_preprocessor_ft.transform(X_test)

# Prepare final fine-tuned parameters
final_ft_params = best_params_clean.copy()
final_ft_params["n_estimators"] = best_n_estimators

# Create the final fine-tuned LightGBM model
final_lgbm_finetuned_model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=42,
    n_jobs=MODEL_JOBS,
    verbosity=-1,
    **final_ft_params
)

# Train the final fine-tuned model on the full training set
final_lgbm_finetuned_model.fit(
    X_train_transformed_ft,
    y_train
)

In [ ]:
# 37. Evaluate final fine-tuned LightGBM model

# Predict class labels on the transformed test set
y_pred_lgbm_finetuned = final_lgbm_finetuned_model.predict(X_test_transformed_ft)

# Predict churn probabilities on the transformed test set
y_proba_lgbm_finetuned = final_lgbm_finetuned_model.predict_proba(X_test_transformed_ft)[:, 1]

# Calculate evaluation metrics
accuracy_lgbm_finetuned = accuracy_score(y_test, y_pred_lgbm_finetuned)
precision_lgbm_finetuned = precision_score(y_test, y_pred_lgbm_finetuned, zero_division=0)
recall_lgbm_finetuned = recall_score(y_test, y_pred_lgbm_finetuned, zero_division=0)
f1_lgbm_finetuned = f1_score(y_test, y_pred_lgbm_finetuned, zero_division=0)
roc_auc_lgbm_finetuned = roc_auc_score(y_test, y_proba_lgbm_finetuned)

print("Final Fine-Tuned LightGBM Results")
print("=" * 50)
print(f"Accuracy:  {accuracy_lgbm_finetuned:.4f}")
print(f"Precision: {precision_lgbm_finetuned:.4f}")
print(f"Recall:    {recall_lgbm_finetuned:.4f}")
print(f"F1-score:  {f1_lgbm_finetuned:.4f}")
print(f"ROC-AUC:   {roc_auc_lgbm_finetuned:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm_finetuned, zero_division=0))

# Confusion matrix
cm_lgbm_finetuned = confusion_matrix(y_test, y_pred_lgbm_finetuned)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm_finetuned,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Final Fine-Tuned LightGBM")
plt.show()

In [ ]:
# 38. Optimize threshold for fine-tuned LightGBM

# Use the validation model from Cell 37 to choose threshold
# This avoids choosing threshold directly from the test set

y_val_proba_finetuned = lgbm_finetuned_model.predict_proba(X_val_ft_transformed)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

finetuned_threshold_results = []

for threshold in thresholds:
    y_val_pred_finetuned = (y_val_proba_finetuned >= threshold).astype(int)

    accuracy = accuracy_score(y_val_ft, y_val_pred_finetuned)
    precision = precision_score(y_val_ft, y_val_pred_finetuned, zero_division=0)
    recall = recall_score(y_val_ft, y_val_pred_finetuned, zero_division=0)
    f1 = f1_score(y_val_ft, y_val_pred_finetuned, zero_division=0)

    finetuned_threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

finetuned_threshold_df = pd.DataFrame(finetuned_threshold_results)

finetuned_threshold_df = finetuned_threshold_df.sort_values(
    by="F1-score",
    ascending=False
)

finetuned_threshold_df.head(10)

In [ ]:
# 39. Evaluate fine-tuned LightGBM with optimized threshold

# Select the best threshold based on validation F1-score
best_finetuned_threshold = finetuned_threshold_df.iloc[0]["Threshold"]

print("Selected threshold for fine-tuned LightGBM:")
print(best_finetuned_threshold)

# Convert test probabilities into class predictions using the selected threshold
y_pred_lgbm_finetuned_threshold = (
    y_proba_lgbm_finetuned >= best_finetuned_threshold
).astype(int)

# Calculate evaluation metrics
accuracy_lgbm_finetuned_threshold = accuracy_score(
    y_test,
    y_pred_lgbm_finetuned_threshold
)

precision_lgbm_finetuned_threshold = precision_score(
    y_test,
    y_pred_lgbm_finetuned_threshold,
    zero_division=0
)

recall_lgbm_finetuned_threshold = recall_score(
    y_test,
    y_pred_lgbm_finetuned_threshold,
    zero_division=0
)

f1_lgbm_finetuned_threshold = f1_score(
    y_test,
    y_pred_lgbm_finetuned_threshold,
    zero_division=0
)

roc_auc_lgbm_finetuned_threshold = roc_auc_score(
    y_test,
    y_proba_lgbm_finetuned
)

print("Fine-Tuned LightGBM with Optimized Threshold Results")
print("=" * 65)
print(f"Threshold:  {best_finetuned_threshold:.2f}")
print(f"Accuracy:   {accuracy_lgbm_finetuned_threshold:.4f}")
print(f"Precision:  {precision_lgbm_finetuned_threshold:.4f}")
print(f"Recall:     {recall_lgbm_finetuned_threshold:.4f}")
print(f"F1-score:   {f1_lgbm_finetuned_threshold:.4f}")
print(f"ROC-AUC:    {roc_auc_lgbm_finetuned_threshold:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lgbm_finetuned_threshold,
    zero_division=0
))

# Confusion matrix
cm_lgbm_finetuned_threshold = confusion_matrix(
    y_test,
    y_pred_lgbm_finetuned_threshold
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm_finetuned_threshold,
    display_labels=["Not Churn", "Churn"]
)

disp.plot()
plt.title("Confusion Matrix - Fine-Tuned LightGBM with Optimized Threshold")
plt.show()

In [ ]:
# 40. Compare final LightGBM models

final_lgbm_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LightGBM Default",
        "LightGBM Tuned",
        f"LightGBM Tuned + Threshold {best_threshold:.2f}",
        "Fine-Tuned LightGBM",
        f"Fine-Tuned LightGBM + Threshold {best_finetuned_threshold:.2f}"
    ],
    "Accuracy": [
        accuracy_logistic,
        accuracy_lgbm,
        accuracy_lgbm_tuned,
        accuracy_lgbm_tuned_threshold,
        accuracy_lgbm_finetuned,
        accuracy_lgbm_finetuned_threshold
    ],
    "Precision": [
        precision_logistic,
        precision_lgbm,
        precision_lgbm_tuned,
        precision_lgbm_tuned_threshold,
        precision_lgbm_finetuned,
        precision_lgbm_finetuned_threshold
    ],
    "Recall": [
        recall_logistic,
        recall_lgbm,
        recall_lgbm_tuned,
        recall_lgbm_tuned_threshold,
        recall_lgbm_finetuned,
        recall_lgbm_finetuned_threshold
    ],
    "F1-score": [
        f1_logistic,
        f1_lgbm,
        f1_lgbm_tuned,
        f1_lgbm_tuned_threshold,
        f1_lgbm_finetuned,
        f1_lgbm_finetuned_threshold
    ],
    "ROC-AUC": [
        roc_auc_logistic,
        roc_auc_lgbm,
        roc_auc_lgbm_tuned,
        roc_auc_lgbm_tuned_threshold,
        roc_auc_lgbm_finetuned,
        roc_auc_lgbm_finetuned_threshold
    ]
})

final_lgbm_comparison.sort_values(
    by="F1-score",
    ascending=False
)

In [ ]:
# Compare observed test scores for this experiment (reporting only)
final_lgbm_comparison_sorted = final_lgbm_comparison.sort_values(
    by="F1-score", ascending=False
)
display(final_lgbm_comparison_sorted)
print("Highest observed test F1:", final_lgbm_comparison_sorted.iloc[0]["Model"])
print("This ranking is descriptive; it does not select the exported model.")

## Interpretation and limitations

Hyperparameter tuning uses cross-validation ROC-AUC on the training set.
Early stopping chooses the number of trees using validation AUC, then the
fine-tuned model is retrained on all training rows. Threshold optimization
uses validation F1; its chosen threshold is computed at runtime, not fixed at 0.60.

The test comparison describes these experiments. The export below always uses
the fine-tuned LightGBM pipeline with its validation-derived threshold, rather
than choosing an export from the test ranking. Fine-tuning is not guaranteed
to outperform the baselines on every dataset or metric.

**Validation limitation inherited from the original notebook:** the inner
validation rows already participated in the earlier hyperparameter search,
and the same validation split is reused for early stopping and threshold
selection. For stricter evaluation, hold out validation before tuning or use
nested CV, then evaluate the chosen strategy once on untouched test data.

In [ ]:
# Export a fixed strategy: fine-tuned LightGBM + validation-selected threshold
# Both components below are already fitted on the full training set.
final_pipeline = Pipeline([
    ("preprocessor", final_preprocessor_ft),
    ("model", final_lgbm_finetuned_model),
])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "final_lightgbm_churn_model.joblib"
threshold_path = ARTIFACT_DIR / "final_churn_threshold.joblib"
joblib.dump(final_pipeline, model_path)
joblib.dump(float(best_finetuned_threshold), threshold_path)
final_lgbm_comparison.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)

# Reload from disk and predict one customer using the matching threshold.
loaded_model = joblib.load(model_path)
loaded_threshold = joblib.load(threshold_path)
sample_customer = X_test.iloc[[0]]
sample_churn_probability = loaded_model.predict_proba(sample_customer)[:, 1][0]
sample_prediction = int(sample_churn_probability >= loaded_threshold)
display(sample_customer)
print(f"Churn probability: {sample_churn_probability:.4f}")
print(f"Decision threshold: {loaded_threshold:.2f}")
print("Prediction:", "Churn" if sample_prediction else "Not churn")
print(f"Artifacts saved to: {ARTIFACT_DIR.resolve()}")